In [ ]:
from monai.transforms import (
    Compose, LoadImaged, EnsureChannelFirstd, Orientationd, Spacingd,
    ScaleIntensityRangePercentilesd, Resized,
    RandFlipd, RandRotate90d, RandGaussianNoised, EnsureTyped,
)
from sklearn.model_selection import train_test_split
import pandas as pd
from monai.data import DataLoader
import torch
from monai.networks.nets import AutoEncoder
import pydicom
pydicom.config.settings.reading_validation_mode = pydicom.config.WARN


In [2]:
df_data = pd.read_csv("./data/labels_duke.csv")
data = df_data.to_dict(orient="records")


train_data, val_data = train_test_split(
    data, test_size=0.2, stratify=[d["label"] for d in data], random_state=42
)

In [ ]:

transforms = Compose([
    LoadImaged(keys="image", reader="PydicomReader"),          
    EnsureChannelFirstd(keys="image"),                     
    Orientationd(keys="image", axcodes="RAS"),            
    Spacingd(keys="image", pixdim=(1.0, 1.0, 1.0),        
             mode="bilinear"),
    ScaleIntensityRangePercentilesd(                       
        keys="image", lower=1, upper=99,
        b_min=0.0, b_max=1.0, clip=True),
    Resized(keys="image", spatial_size=(96, 96, 96)),      
    RandFlipd(keys="image", prob=0.5, spatial_axis=0),    
    RandRotate90d(keys="image", prob=0.5, spatial_axes=(0, 1)),
    RandGaussianNoised(keys="image", prob=0.2),
    EnsureTyped(keys="image"),
])



/home/amiyazono/miniconda3/envs/monai/lib/python3.14/site-packages/monai/utils/deprecate_utils.py:321: FutureWarning: monai.transforms.spatial.dictionary Orientationd.__init__:labels: Current default value of argument `labels=(('L', 'R'), ('P', 'A'), ('I', 'S'))` was changed in version None from `labels=(('L', 'R'), ('P', 'A'), ('I', 'S'))` to `labels=None`. Default value changed to None meaning that the transform now uses the 'space' of a meta-tensor, if applicable, to determine appropriate axis labels.
  warn_deprecated(argname, msg, warning_category)


In [4]:
from monai.data import CacheDataset
import torch.multiprocessing as mp

try:
    mp.set_start_method("spawn", force=True)
except RuntimeError:
    pass


ds = CacheDataset(data, transform=transforms, cache_rate=1.0, num_workers=8)


Loading dataset: 100%|██████████| 938/938 [11:20<00:00,  1.38it/s]


In [ ]:
loader = DataLoader(ds, batch_size=10, shuffle=True, num_workers=4)

device = torch.device("cuda")
model = AutoEncoder(
    spatial_dims=3,
    in_channels=1,
    out_channels=1,
    channels=(16, 32, 64, 128),
    strides=(2, 2, 2, 2),
    inter_channels=(256,),        
    inter_dilations=(1,),
    num_res_units=2,
).to(device)

opt = torch.optim.Adam(model.parameters(), 1e-3)
loss_fn = torch.nn.MSELoss()

In [6]:
model.train()
for epoch in range(50):
    for batch in loader:
        x = batch["image"].to(device)
        opt.zero_grad()
        recon = model(x)
        loss = loss_fn(recon, x)
        loss.backward()
        opt.step()
    print(epoch, loss.item())

0 0.015409756451845169
1 0.008170709945261478


2 0.006084555760025978
3 0.006502982694655657
4 0.00436353450641036
5 0.004644240718334913
6 0.005037346389144659
7 0.0038477214984595776
8 0.0037253075279295444
9 0.004762932192534208
10 0.0036930886562913656
11 0.0028128991834819317
12 0.003600483760237694
13 0.002975915325805545
14 0.004966724663972855
15 0.0031747068278491497
16 0.005598145537078381
17 0.003420958062633872
18 0.0024851830676198006
19 0.002716650255024433
20 0.004057095851749182
21 0.0029454082250595093
22 0.0031343053560703993
23 0.00313097657635808
24 0.00317285000346601
25 0.0024615295697003603
26 0.0022098380140960217
27 0.002129553584381938
28 0.002655548043549061
29 0.0035281809978187084
30 0.0025415183044970036
31 0.0031589434947818518
32 0.0022918442264199257
33 0.002138234907761216
34 0.0023872400633990765
35 0.0019052509451285005
36 0.002399775432422757
37 0.0017957397503778338
38 0.0021098805591464043
39 0.0031377372797578573
40 0.0017653057584539056
41 0.0026284519117325544
42 0.002032895339652896
43 0.0